## Test from outside 

this section is created for testing from outside of used database (UID2021) 

this is a test for single picture, if you have bunch ,you need to adjust accordingly.(you can check the way we created dataset from UID2021)

In [7]:
# u need to adjust the image path 

image_path = 'ERI_BG_9_CBF.png'

## Extracting the features:
in this section you extract the features

In [8]:
import numpy as np
import cv2
from skimage import filters, metrics
from skimage import io, img_as_float, color
from skimage.filters import gaussian, sobel,laplace
from skimage.restoration import denoise_tv_chambolle
from skimage import io, color, morphology, exposure
from scipy.fft import fft2, ifft2, fftshift, ifftshift
from skimage import img_as_ubyte
from skimage.feature import local_binary_pattern
import matplotlib.pyplot as plt
from scipy.special import gamma
from scipy.optimize import minimize
import csv
import pywt

def local_contrast_luminance(image, block_size=32):
    # Convert the image to float32 if it is not already
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)
    
    # Convert to grayscale if the image is in RGB format
    if len(image.shape) == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image

    height, width = gray_image.shape

    contrast_values = []
    local_lum = []
    # Iterate over each block
    for y in range(0, height, block_size):
        for x in range(0, width, block_size):
            # Define the block
            block = gray_image[y:y+block_size, x:x+block_size]
            # Check if the block is valid
            if block.shape[0] != block_size or block.shape[1] != block_size:
                continue
            
            # Compute mean luminance
            block_mean = np.mean(block)
            # Compute local contrast (standard deviation in this case)
            contrast = np.std(block)
            contrast_values.append(contrast)

    # Return the average contrast across all blocks
    return np.var(contrast_values)


def image_entropy(image):
    # Convert image to grayscale if it's not already
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Ensure the image is in uint8 format
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)

    # Calculate histogram
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist = hist / hist.sum()  # Normalize the histogram

    # Calculate entropy
    entropy = -np.sum(hist * np.log2(hist + 1e-10))  # Add small value to avoid log(0)

    return entropy

# contrast 
def apply_ahe(image, clip_limit=0.05):# adaptive histogram 
    ahe_image = exposure.equalize_adapthist(image, clip_limit=clip_limit)
    return ahe_image


# Function to calculate MSCN coefficients at multiple scales
def calculate_mscn_multi_scale(image, scales):
    mscn_coeffs = []
    
    for scale in scales:
        sigma = scale * 7 / 6
        mu = gaussian(image, sigma=sigma)
        var = gaussian(image**2, sigma=sigma) - mu**2
        sigma = np.sqrt(np.maximum(var, 1e-10))  # Using np.maximum to avoid negative values
        C = 1.0 / 255  # Small constant to prevent division by zero
        mscn = (image - mu) / (sigma + C)
        mscn_coeffs.append(mscn)
    
    return mscn_coeffs

# Function to estimate AGGD parameters from a vector
def estimate_aggd_params(vec):
    left_std = np.sqrt(np.mean((vec[vec < 0])**2))
    right_std = np.sqrt(np.mean((vec[vec > 0])**2))
    gamma_hat = left_std / right_std if right_std != 0 else 0
    
    rhat = (np.mean(np.abs(vec)))**2 / np.mean(vec**2)
    rhat_norm = (gamma(2 / np.mean(np.abs(vec))) / gamma(1 / np.mean(np.abs(vec))))**2
    
    agg_param = np.zeros(2)
    
    agg_param[0] = np.sqrt((rhat / rhat_norm) * gamma_hat)
    agg_param[1] = np.sqrt((rhat / rhat_norm) / gamma_hat)
    
    return agg_param

# Function to fit AGGD to MSCN coefficients
def fit_aggd(mscn):
    alpha = estimate_aggd_params(mscn)
    beta_l = np.sqrt(np.mean((mscn[mscn < 0])**2))
    beta_r = np.sqrt(np.mean((mscn[mscn > 0])**2))
    return alpha[0], alpha[1], beta_l**2, beta_r**2
#ggd method : 
def estimate_ggd_params(vec):
    def ggd_loss(params, vec):
        alpha, sigma = params
        beta = sigma * np.sqrt(gamma(1 / alpha) / gamma(3 / alpha))
        likelihoods = (alpha / (2 * beta * gamma(1 / alpha))) * np.exp(-(np.abs(vec) / beta) ** alpha)
        return -np.sum(np.log(likelihoods + 1e-10))  # Add small value to avoid log(0)
    
    initial_params = [2.0, np.std(vec)]  # Initial guess for alpha and sigma
    result = minimize(ggd_loss, initial_params, args=(vec,), bounds=[(0.1, 10), (1e-3, 1e3)])
    alpha, sigma = result.x
    return alpha, sigma

# Function to fit GGD to MSCN coefficients
def fit_ggd(mscn):
    alpha, sigma = estimate_ggd_params(mscn)
    return alpha, sigma

# Function to analyze MSCN coefficients with GGD
def analyze_mscn_coeffs_ggd(mscn_coeffs):
    ggd_params = []
    for mscn in mscn_coeffs:
        ggd_params.append(fit_ggd(mscn))
    return ggd_params

# Function to analyze MSCN coefficients
def analyze_mscn_coeffs_aggd(mscn_coeffs):
    agg_params = []
    
    for mscn in mscn_coeffs:
        agg_params.append(fit_aggd(mscn))
    
    return agg_params


# frequence domain with high pass filtering sharpness 

def high_pass_filter(size, cutoff):

    rows, cols = size
    crow, ccol = rows // 2, cols // 2
    filter = np.ones((rows, cols), np.float32)
    r, c = np.ogrid[:rows, :cols]
    distance = np.sqrt((r - crow) ** 2 + (c - ccol) ** 2)
    filter[distance <= cutoff] = 0
    return filter

def frequency_domain_sharpness(image, cutoff):

    # Convert image to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_image = np.float32(gray_image)
    
    # Perform FFT
    f_transform = fft2(gray_image)
    f_transform_shifted = fftshift(f_transform)
    
    # Create high-pass filter
    rows, cols = gray_image.shape
    hp_filter = high_pass_filter((rows, cols), cutoff)
    
    # Apply high-pass filter
    filtered_transform = f_transform_shifted * hp_filter
    filtered_image = np.abs(ifft2(ifftshift(filtered_transform)))
    
    # Compute sharpness as the variance of the high-pass filtered image
    sharpness = np.var(filtered_image)
    return sharpness
# Edge Detection-Based Sharpness

def canny_sharpness(image, sigma, threshold1, threshold2):
    if len(image.shape) == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image
    
    blurred_image = cv2.GaussianBlur(gray_image, (0, 0), sigmaX=sigma)
    edges = cv2.Canny(blurred_image, threshold1, threshold2)
    sharpness = np.var(edges)
    
    return sharpness


# colorfullness and color attributes:
def compute_color_moments(image):
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    L, a, b = cv2.split(lab_image)
    moments = {}
    for channel, name in zip([L, a, b], ['L', 'a', 'b']):
        mean = np.mean(channel)
        variance = np.var(channel)
        skewness = np.mean((channel - mean) ** 3) / (np.std(channel) ** 3)
        moments[name] = {
            'mean': mean,
            'variance': variance,
            'skewness': skewness
        }
    return moments


def convert_image(image):
    # Convert the image to 8-bit unsigned integer if it's in floating-point format
    if image.dtype == np.float64 or image.dtype == np.float32:
        image = img_as_ubyte(image)
    
    # Check if the image has an alpha channel (4 channels)
    if image.shape[2] == 4:
        # Remove the alpha channel
        image = image[:, :, :3]
    
    # Convert the image from RGB to BGR
    image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    return image_bgr
data_set = []


image_rgb = img_as_float(io.imread(image_path))
if image_rgb.shape[2] == 4:
    image_rgb = image_rgb[:, :, :3]
image_bgr = convert_image(image_rgb)
height,width ,_ = image_rgb.shape
image_gray = color.rgb2gray(image_rgb)
adh_image = apply_ahe(image_gray)
mscn_coeffs = calculate_mscn_multi_scale(image_gray, scales=[1,2,3])#aggd
aggd_params = analyze_mscn_coeffs_aggd(mscn_coeffs)
mscn_coeffs_adh = calculate_mscn_multi_scale(adh_image, scales=[1,2,3])
aggd_params_adh = analyze_mscn_coeffs_aggd(mscn_coeffs_adh)
ggd_params = analyze_mscn_coeffs_ggd(mscn_coeffs)
ggd_params_adh = analyze_mscn_coeffs_ggd(mscn_coeffs_adh)
dc = local_contrast_luminance(image_rgb)
entropy = image_entropy(image_gray)
freqcut10 = frequency_domain_sharpness(image_bgr,10)
freqcut40 = frequency_domain_sharpness(image_bgr,40)
cn_sharp_strong = canny_sharpness(image_bgr,0.5 , 150 , 200)
moment = compute_color_moments(image_bgr)
list1 = [dc,entropy,freqcut10,freqcut40,cn_sharp_strong]
for params in aggd_params:
    AL_adh, AR_adh, BL_adh, BR_adh = params
    list1.extend([AL_adh, AR_adh, BL_adh, BR_adh])
    
for params in ggd_params_adh:
    Ap_h,sg_h = params
    list1.extend([Ap_h,sg_h])
    
for k in moment.keys():
    a = moment[k]
    b = a['mean']
    c = a['variance']
    d = a['skewness']
    list1.extend([b,c,d])
data_set.append(list1)


with open('data_set_single.csv', 'w', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow([ 'lum_l', 'br_ent', 'freq1','freq3', 'cannysharp1', 
    'AL_adh1', 'AR_adh1', 'BL_adh1', 'BR_adh1', 'AL_adh2', 'AR_adh2', 'BL_adh2', 'BR_adh2', 'AL_adh3', 'AR_adh3', 'BL_adh3', 'BR_adh3',
    'Ap_h1', 'sg_h1', 'Ap_h2', 'sg_h2', 'Ap_h3', 'sg_h3',
    'L_mean', 'L_variance', 'L_skewness', 'a_mean', 'a_variance', 'a_skewness', 
    'b_mean', 'b_variance', 'b_skewness'])
    writer.writerows(data_set)



## Test:
* Provide the extracted features to the machine, and it will generate the result. For a single test, the program is designed to display a picture as well.
* If your test subject has a score, you can either load it or input it manually. That part of the code has been commented out, but you can enable it as needed.

In [ ]:
import matplotlib.pyplot as plt
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
# model is already leaned and it will be loaded

#load the picture 

input = pd.read_csv('data_set_single.csv')
# input_target = pd.read_csv('')

# Load the learned model
learned_estimator = joblib.load('learned_gpr_model.pkl')
# Load scaler
scaler = joblib.load('scaler.pkl')

# Path to the picture (ensure it's in the same directory as the script)
picture_to_show = 'ERI_BG_9_CBF.png'

input_data_transformed = scaler.transform(input)  # Transform the input data

# Predict using the learned method
predicted_results = learned_estimator.predict(input_data_transformed)

# Combine the real targets and predicted results into a DataFrame
results = pd.DataFrame({
    # 'Real Target':input_target
    'Predicted Target': predicted_results
})

# Save the combined results to a CSV file
results_file = 'input_predictions_outsidedata.csv'
results.to_csv(results_file, index=False)

# Display the image 
# this is only for simple test , modify for bunch . 
image = plt.imread(picture_to_show)  # Read the image
plt.imshow(image)  # Show the image

#for adding the target value to picture
#plt.title(f'real_score :{input_target},Predicted_score :{predicted_results[0]: .3f} ')

plt.title(f'Predicted_score :{predicted_results[0]: .3f} ')
plt.axis('off')  # Hide axes
plt.show()

print(results)


